# Week 9: 网络优化

## 学习目标

1. 理解图论的基本概念
2. 掌握最短路、最大流等经典问题
3. 学会使用 NetworkX 库
4. 应用网络优化解决实际问题

## 1. 图论基础

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

print("网络优化工具已加载")

### 1.1 图的基本概念

- **顶点（Vertex）**：网络中的节点
- **边（Edge）**：节点之间的连接
- **权重（Weight）**：边的成本或容量
- **有向图 vs 无向图**：边是否有方向

In [ ]:
# 创建简单图
G = nx.Graph()  # 无向图

# 添加节点
G.add_nodes_from([1, 2, 3, 4, 5])

# 添加边
G.add_edges_from([(1, 2), (1, 3), (2, 3), (2, 4), (3, 4), (4, 5)])

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 无向图
nx.draw(G, with_labels=True, node_color='lightblue', 
        node_size=500, font_size=16, ax=axes[0])
axes[0].set_title('无向图')

# 有向图
DG = nx.DiGraph(G)
nx.draw(DG, with_labels=True, node_color='lightgreen',
        node_size=500, font_size=16, ax=axes[1],
        arrowsize=20)
axes[1].set_title('有向图')

plt.tight_layout()
plt.show()

# 基本属性
print("图的基本属性")
print("=" * 40)
print(f"节点数: {G.number_of_nodes()}")
print(f"边数: {G.number_of_edges()}")
print(f"平均度: {np.mean([d for n, d in G.degree()]):.2f}")

## 2. 最短路问题

### 2.1 单车配送路径

In [ ]:
# 创建带权重的图（城市间距离）
G = nx.DiGraph()

# 添加边（起点，终点，距离）
edges = [
    ('仓库', 'A', 5), ('仓库', 'B', 3),
    ('A', 'B', 2), ('A', 'C', 4),
    ('B', 'C', 6), ('B', 'D', 3),
    ('C', 'D', 2), ('C', '目的地', 5),
    ('D', '目的地', 4)
]

for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# 最短路径
path = nx.dijkstra_path(G, '仓库', '目的地')
distance = nx.dijkstra_path_length(G, '仓库', '目的地')

print("最短配送路径")
print("=" * 40)
print(f"路径: {' -> '.join(path)}")
print(f"总距离: {distance} 公里")

In [ ]:
# 可视化
fig, ax = plt.subplots(figsize=(12, 8))

# 布局
pos = nx.spring_layout(G, seed=42)

# 绘制所有边
nx.draw_networkx_edges(G, pos, ax=ax, edge_color='gray', width=1)

# 绘制最短路径
path_edges = list(zip(path[:-1], path[1:]))
nx.draw_networkx_edges(G, pos, edgelist=path_edges, 
                       edge_color='red', width=3)

# 绘制节点
nx.draw_networkx_nodes(G, pos, ax=ax, node_color='lightblue',
                       node_size=700)

# 标注节点
nx.draw_networkx_labels(G, pos, ax=ax, font_size=12)

# 标注边权重
edge_labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels, ax=ax)

ax.set_title(f'最短配送路径: {" -> ".join(path)}\n总距离: {distance} 公里')
ax.axis('off')

plt.tight_layout()
plt.show()

### 2.2 所有节点对的最短路径

In [ ]:
# 计算所有节点对之间的最短距离
all_pairs = dict(nx.all_pairs_dijkstra_path_length(G))

# 转为 DataFrame
nodes = list(G.nodes())
distance_matrix = pd.DataFrame(index=nodes, columns=nodes)

for source in nodes:
    for target in nodes:
        if source == target:
            distance_matrix.loc[source, target] = 0
        elif target in all_pairs[source]:
            distance_matrix.loc[source, target] = all_pairs[source][target]
        else:
            distance_matrix.loc[source, target] = np.inf

print("所有节点对的最短距离")
print("=" * 40)
print(distance_matrix)

## 3. 最大流问题

In [ ]:
# 单车调度流量问题
# 目标：最大化从调配中心到需求站点的车辆流量

G = nx.DiGraph()

# 添加边（起点，终点，容量）
edges = [
    ('调配中心', 'A', 20), ('调配中心', 'B', 15),
    ('A', 'C', 10), ('A', 'D', 12),
    ('B', 'C', 8), ('B', 'D', 10),
    ('C', '站点1', 15), ('C', '站点2', 8),
    ('D', '站点1', 10), ('D', '站点2', 12)
]

for u, v, c in edges:
    G.add_edge(u, v, capacity=c)

# 计算最大流
flow_value, flow_dict = nx.maximum_flow(G, '调配中心', '站点1')

print("到站点1的最大流量")
print("=" * 40)
print(f"最大流量: {flow_value} 辆")

print("\n各路径流量分配:")
for u in flow_dict:
    for v, flow in flow_dict[u].items():
        if flow > 0:
            print(f"  {u} -> {v}: {flow} 辆")

In [ ]:
# 可视化网络流
fig, ax = plt.subplots(figsize=(12, 8))

pos = {
    '调配中心': (0, 1),
    'A': (1, 1.5), 'B': (1, 0.5),
    'C': (2, 1.5), 'D': (2, 0.5),
    '站点1': (3, 1)
}

# 绘制边
for u, v in G.edges():
    capacity = G[u][v]['capacity']
    flow = flow_dict[u][v] if v in flow_dict[u] else 0
    
    # 边宽度和颜色根据流量
    width = 1 + flow / 5
    color = plt.cm.Blues(flow / capacity)
    
    nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], 
                          width=width, edge_color=[color], ax=ax,
                          arrowsize=20)
    
    # 标注流量/容量
n    mid_x = (pos[u][0] + pos[v][0]) / 2
    mid_y = (pos[u][1] + pos[v][1]) / 2
    ax.text(mid_x, mid_y, f"{flow}/{capacity}", fontsize=10, 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

# 绘制节点
nx.draw_networkx_nodes(G, pos, ax=ax, node_color='lightgreen',
                       node_size=800)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=11)

ax.set_title(f'单车调配网络流\n最大流量: {flow_value} 辆')
ax.axis('off')

plt.tight_layout()
plt.show()

## 4. 最小生成树

In [ ]:
# 问题：用最小成本连接所有站点

G = nx.Graph()

stations = ['S1', 'S2', 'S3', 'S4', 'S5']
edges = [
    ('S1', 'S2', 4), ('S1', 'S3', 2),
    ('S2', 'S3', 1), ('S2', 'S4', 5),
    ('S3', 'S4', 8), ('S3', 'S5', 10),
    ('S4', 'S5', 3)
]

for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# 最小生成树
mst = nx.minimum_spanning_tree(G)
mst_weight = mst.size(weight='weight')

print("最小生成树")
print("=" * 40)
print("选择的边:")
for u, v, w in mst.edges(data='weight'):
    print(f"  {u} - {v}: 成本 {w}")
print(f"\n总成本: {mst_weight}")

In [ ]:
# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pos = nx.spring_layout(G, seed=42)

# 原图
nx.draw(G, pos, with_labels=True, node_color='lightblue',
        node_size=500, font_size=12, ax=axes[0])
edge_labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels, ax=axes[0])
axes[0].set_title('原始网络')

# 最小生成树
nx.draw(mst, pos, with_labels=True, node_color='lightgreen',
        node_size=500, font_size=12, ax=axes[1], width=3)
edge_labels_mst = nx.get_edge_attributes(mst, 'weight')
nx.draw_networkx_edge_labels(mst, pos, edge_labels_mst, ax=axes[1])
axes[1].set_title(f'最小生成树 (总成本: {mst_weight})')

plt.tight_layout()
plt.show()

## 5. 网络中心性分析

In [ ]:
# 创建单车站点网络
G = nx.Graph()

# 添加站点和连接
stations = ['A001', 'A002', 'A003', 'B001', 'B002', 'C001']
connections = [
    ('A001', 'A002'), ('A001', 'A003'),
    ('A002', 'A003'), ('A002', 'B001'),
    ('A003', 'B002'), ('B001', 'B002'),
    ('B001', 'C001'), ('B002', 'C001')
]

G.add_nodes_from(stations)
G.add_edges_from(connections)

# 计算中心性指标
degree_cent = nx.degree_centrality(G)
betweenness_cent = nx.betweenness_centrality(G)
closeness_cent = nx.closeness_centrality(G)

# 结果汇总
centrality_df = pd.DataFrame({
    '站点': stations,
    '度中心性': [degree_cent[s] for s in stations],
    '中介中心性': [betweenness_cent[s] for s in stations],
    '接近中心性': [closeness_cent[s] for s in stations]
})

print("站点中心性分析")
print("=" * 40)
print(centrality_df.to_string(index=False))

In [ ]:
# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pos = nx.spring_layout(G, seed=42)

metrics = [
    ('度中心性', degree_cent),
    ('中介中心性', betweenness_cent),
    ('接近中心性', closeness_cent)
]

for i, (title, centrality) in enumerate(metrics):
    # 节点大小根据中心性
    sizes = [500 + 2000 * centrality[s] for s in stations]
    
    nx.draw(G, pos, with_labels=True, node_size=sizes,
            node_color=[centrality[s] for s in stations],
            cmap=plt.cm.Reds, ax=axes[i])
    
    axes[i].set_title(title)

plt.tight_layout()
plt.show()

## 6. Research Thinking

### 问题 1：网络 vs 线性规划

什么时候用网络模型？什么时候用线性规划？

**回答：**

- **网络模型**：问题有明确的结构（路径、流量）
- **线性规划**：更通用的优化问题
- **关系**：网络问题是线性规划的特殊形式，有更高效的算法

### 问题 2：实际网络的不确定性

实际网络中边权重可能变化，如何处理？

**回答：**

1. **随机网络**：边权重服从概率分布
2. **鲁棒优化**：考虑最坏情况
3. **动态规划**：权重随时间变化
4. **实时更新**：根据新信息重新计算

### 问题 3：网络规模

大规模网络如何处理？

**回答：**

- **分层网络**：先在高层优化，再细化
- **区域分解**：将网络分成子网络
- **启发式算法**：找到近优解
- **并行计算**：加速计算过程

## 7. 练习

### 练习 1
设计一个车辆路径规划问题（VRP 简化版）。

In [ ]:
# 你的代码


### 练习 2
分析一个真实城市的地铁网络。

In [ ]:
# 你的代码


### 练习 3
实现一个二分图匹配问题。

In [ ]:
# 你的代码


## 8. 总结

### 本周学习要点

1. **图论基础**：顶点、边、权重、方向
2. **最短路问题**：Dijkstra 算法
3. **最大流问题**：流量分配优化
4. **最小生成树**：最小成本连接
5. **网络中心性**：节点重要性分析

### 关键洞察

- 网络模型适合结构化的优化问题
- NetworkX 提供了丰富的图算法
- 中心性分析有助于识别关键节点
- 实际应用需要考虑不确定性